# WebSweep Example for Researchers

This notebook is split into two parts:

- Part A: the smallest default workflow (crawler -> extractor -> consolidator)
- Part B: extended usage patterns (one-pass mode, custom `FileExtractor`, and parameter defaults)

```text
Input:
  URL list
      |
      v
  [Crawler]
    In:  URL list
    Out: crawled_data/*.zip + overview_urls.{duckdb|db|tsv}
      |
      v
  [Extractor]
    In:  overview file + crawled_data/*.zip
    Out: extracted_data/*.ndjson
      |
      v
  [Consolidator]
    In:  extracted_data/*.ndjson
    Out: consolidated_data/*.ndjson (domain-level records)
```


In [1]:
from pathlib import Path
import json

urls = [
    "https://www.dggrootverbruik.nl/",
    "https://www.gosliga.nl/",
    "https://www.heeren2.nl/",
]

run_dir = Path("./data")
output_dir = run_dir / "research_output"
output_dir.mkdir(parents=True, exist_ok=True)

print("urls:", urls)
print("output_dir:", output_dir)


urls: ['https://www.dggrootverbruik.nl/', 'https://www.gosliga.nl/', 'https://www.heeren2.nl/']
output_dir: data/research_output


## Part A. Default Workflow (No Extra Parameters)

### Step 1. Crawl (defaults)

**Input**
- `urls`

**Output**
- `crawled_data/*.zip`
- `overview_urls.{duckdb|db|tsv}`


In [2]:
from websweep.crawler.crawler import Crawler

crawler = Crawler(target_folder_path=output_dir)
crawler.crawl_base_urls(urls)


100%|█████████████████████████████████████████████| 3/3 [01:00<00:00, 20.11s/it]

Crawled 17 pages from 3 urls to level 3 in 62.5 seconds.


In [8]:
!ls data/research_output/crawled_data

dggrootverbruik.nl.zip gosliga.nl.zip         heeren2.nl.zip


### Step 2. Extract (defaults)

**Input**
- `overview_urls.*`
- `crawled_data/*.zip`

**Output**
- `extracted_data/*.ndjson`


In [4]:
from websweep.extractor.extractor import Extractor

extractor = Extractor(target_folder_path=output_dir)
extractor.extract_urls()


100%|██████████████████████████████████████████| 17/17 [00:00<00:00, 173.94it/s]

Extracted data from 17 pages (0 errors) in 0.1 seconds.


In [13]:
extracted_files = sorted((output_dir / "extracted_data").glob("*.ndjson"))
print("extracted files:", [f.name for f in extracted_files])

test_extracted = extracted_files[0].as_posix()
!head $test_extracted

extracted files: ['extracted_data_2026-02-23_0-1000000.ndjson']
{"domain":"gosliga.nl","identifier":"gosliga.nl","level":0,"website":"https://www.gosliga.nl/","date":"2026-02-23","path":"data/research_output/crawled_data/gosliga.nl/gosliga.nl/2026-02-23/www.gosliga.nl","meta_encoding":"UTF-8","meta_viewport":"width=device-width, initial-scale=1","zipcode":["9061 AE"],"address":["Rinia van Nautaweg 12"],"text":"Gosliga Transportbedrijf BV\nHome\nOver ons\nVacatures\nTransport\nSilovervoer\nKoel- vriestransport\nOpslag of loodsen\nContact\nSilovervoer\nAan het vervoer van levensmiddelen en grondstoffen voor de levensmiddelen- en diervoederindustrie worden speciale eisen gesteld. Voor elk soort...\nKoel- vriestransport\nBehalve het (tijdelijk) opslaan van koel- en vriesproducten voor derden, vervoert Gosliga BV -conform de richtlijnen van HACCP - gekoelde of...\nOpslag of loodsen\nOp industrieterrein ‘Schenkenschans’ in Leeuwarden beschikt Gosliga BV over een koel- en vrieshuis. Hier kunn

### Step 3. Consolidate (defaults)

**Input**
- `extracted_data/*.ndjson`

**Output**
- `consolidated_data/consolidated.ndjson`


In [19]:
from websweep.consolidator.consolidator import Consolidator

input_file = sorted((output_dir / "extracted_data").glob("*.ndjson"))[0]
consolidated_path = output_dir / "consolidated_data" / "consolidated.ndjson"

#consolidated_path.parent.mkdir(parents=True, exist_ok=True)
#consolidated_path.touch(exist_ok=True)

consolidator = Consolidator(str(input_file))
consolidator.consolidate(str(consolidated_path))

print("consolidated file:", consolidated_path)

!cat data/research_output/consolidated_data/consolidated.ndjson

100%|█████████████████████████████████████████| 17/17 [00:00<00:00, 8265.12it/s]

consolidated file: data/research_output/consolidated_data/consolidated.ndjson
{"domain":"gosliga.nl","identifier":"gosliga.nl","phone":{},"email":{},"fax":{},"zipcode":{"9061 AE":11},"address":{"Rinia van Nautaweg 12":11},"kvk":{},"btw":{},"text":" Gosliga Transportbedrijf BV\nHome\nOver ons\nVacatures\nTransport\nSilovervoer\nKoel- vriestransport\nOpslag of loodsen\nContact\nSilovervoer\nAan het vervoer van levensmiddelen en grondstoffen voor de levensmiddelen- en diervoederindustrie worden speciale eisen gesteld. Voor elk soort...\nKoel- vriestransport\nBehalve het (tijdelijk) opslaan van koel- en vriesproducten voor derden, vervoert Gosliga BV -conform de richtlijnen van HACCP - gekoelde of...\nOpslag of loodsen\nOp industrieterrein ‘Schenkenschans’ in Leeuwarden beschikt Gosliga BV over een koel- en vrieshuis. Hier kunnen relaties permanent of juist...\nNa meer dan 100 jaar nog springlevend\nGosliga Transportbedrijf BV\nIn de positieve betekenis is Gosliga BV nog een onderneming ‘v

In [ ]:
if consolidated_path.exists():
    with consolidated_path.open("r", encoding="utf-8") as f:
        first_domain_record = json.loads(f.readline())
    print("consolidated keys:", sorted(first_domain_record.keys()))
    print("example domain:", first_domain_record.get("domain"))


## Part B. Extended Usage

This section shows optional advanced patterns after you understand the default loop.


### B1. Crawl + Extract in One Pass (save disk) + Extension Filters

Use one-pass mode when you want to skip saving raw HTML zip files.
You can also pass `allow_extensions` / `block_extensions` to `Crawler` to control
which linked file types are followed.


In [ ]:
from websweep.crawler.crawler import Crawler

one_pass_dir = run_dir / "research_output_one_pass"
one_pass_dir.mkdir(parents=True, exist_ok=True)

# Optional extension controls (comma-separated string or list both work).
# Keep PDFs and PNGs discoverable, while skipping common binary/archive types.
one_pass_crawler = Crawler(
    target_folder_path=one_pass_dir,
    extract=True,
    save_html=False,
    allow_extensions="pdf,png",
    block_extensions="zip,gz,rar,jpg,jpeg",
)
one_pass_crawler.crawl_base_urls(urls)


### B2. Custom `FileExtractor`

By default, the extractor keeps conservative fields. You can add your own fields by subclassing `FileExtractor`.


In [ ]:
import re
from websweep.extractor.extractor import Extractor, FileExtractor

class ResearchFileExtractor(FileExtractor):
    def _extract_fax(self) -> list:
        pattern = re.compile(
            r"(?is)\b(?:faxnumber|fax|f)\b[^0-9\+]{0,12}"
            r"([\+]?[0-9][0-9\-\s\(\)]{7,20})\b"
        )
        return sorted({m.strip() for m in re.findall(pattern, str(self.soup))})

custom_extractor = Extractor(
    target_folder_path=output_dir,
    file_extractor=ResearchFileExtractor,
)
# custom_extractor.extract_urls()  # uncomment to run custom extraction


### B3. Show Default Function Parameters

This is useful when you want to understand defaults such as `max_level`, `threads_download`, and extractor/consolidator defaults.


In [ ]:
import inspect
from websweep.crawler.crawler import Crawler
from websweep.extractor.extractor import Extractor
from websweep.consolidator.consolidator import Consolidator

print("Crawler.__init__ defaults:")
print(inspect.signature(Crawler.__init__))
print()
print("Extractor.__init__ defaults:")
print(inspect.signature(Extractor.__init__))
print()
print("Consolidator.__init__ defaults:")
print(inspect.signature(Consolidator.__init__))


### Optional add-on module

Repository add-on example:

- `addons/firmbackbone_extractor.py`
